## Lab 03 / Step 3 : Supervised Learning (SL) of Reward Model (RM)

## Task : From a specific prompt, rank positive and negative responses

## Use pre-trained SFT-LLM model from Step 2 

## Freeze it and use it as the backbone for training the reward model

### Xavier Bresson

### Number of data points for GPT-3, 175B parameters
+ Step #1 : 300B tokens
+ Step #2 : 10K-100k pairs (prompt, response)
+ Step #3 : 100K-1M triples (prompt, positive response, negative response)
+ Step #4 : 10K-100K prompts

### Number of data points for this tutorial
+ Step #1 : 200M tokens
+ Step #2 : 200K pairs (prompt, response)
+ Step #3 : 100K triples (prompt, positive response, negative response)
+ Step #4 : 100 prompts

### Objectives
+ Supervised learning of reward
+ Freeze a LLM network to use as backbone for reward prediction
+ Train with batch of pairs (prompt, positive response, negative) for fast training with GPU


In [1]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/IPAM_Mar26_codes/labs_lecture09'
    print(path_to_file)
    # move to Google Drive directory
    os.chdir(path_to_file)
    !pwd
    

In [1]:
# Libraries
import torch
print(torch.__version__)
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.CRITICAL) # remove warnings
import os, datetime


2.1.2


## Time stamp for save/load data


In [2]:
# save time stamp
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")

# check dataset folder exists
data_dir = os.path.join('dataset')
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# select a time stamp
use_saved_time_stamp = False
use_saved_time_stamp = True
if use_saved_time_stamp:
    time_stamp = '26-02-07--18-48-32' # trained on GPU on xxx

print('time_stamp:', time_stamp, '\n')


time_stamp: 26-02-07--18-48-32 



## Load dictionary of tokens from Step 1


In [3]:
# load_file_dictionary = 'dataset/step1_SSL_dictionary_' + str(time_stamp) + '.pt'
file_dictionary = 'dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt' # GPU pre-trained
print('file_dictionary:', file_dictionary, '\n')
import os
os.makedirs('dataset', exist_ok=True)
if not os.path.exists(file_dictionary):
    print(f'Downloading {file_dictionary} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/jvb5xkk0kfzwrm7vzha3o/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt?rlkey=tprax7nqqpvogo04fhe7m2okd&dl=1"
    !wget -q "{dropbox_url}" -O "{file_dictionary}"
dictionary, num_tokens, token2index, index2token = torch.load(file_dictionary) # load dictionary of tokens
print('dictionary:',dictionary,'\n')
print('num_tokens (unique):',num_tokens,'\n')
print('token2index:', token2index,'\n')
print('index2token:', index2token,'\n')
func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'


file_dictionary: dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt 

dictionary: ['51', '57', '63', '69', '75', '81', '87', '93', '99', '<SEP>', '91', '1', '2', '3', '4', '5', '6', '7', '8', '9', '77', '79', '83', '85', '89', '48', '55', '62', '76', '90', '97', '29', '38', '47', '56', '65', '74', '92', '94', '100', '50', '64', '71', '78', '39', '95', '46', '53', '60', '67', '88', '13', '22', '31', '40', '49', '58', '41', '42', '43', '44', '30', '54', '15', '36', '72', '86', '96', '18', '19', '20', '21', '23', '24', '25', '26', '27', '28', '73', '98', '82', '52', '34', '66', '70', '68', '84', '37', '61', '11', '14', '17', '32', '59', '10', '12', '16', '80', '35', '33', '45', '0', 'generate', 'an', 'arithmetic', 'series', 'with', 'terms', 'starting', 'value', 'and', 'common', 'difference', 'Let', 'be', 'the', 'number', 'of', 'then', 'write', 'make', 'a', 'type', 'which', 'starts', 'at', 'elements', '<PAD>', '<EOS>'] 

num_tokens (unique): 129 

token2index: {'51': 0, '57': 1, '63':

## Load pre-trained SFT-LLM network (from Step 2)


In [4]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# GPU training
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length))==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

# class of supervised fine-tuning LM network (step 2)
class SFT_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.seq_pos_encoding = torch.arange(context_length, device=device) # positional encoding = {0,1,2,...,context_length-1}
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
    # Note : No forward function is needed

# load pre-trained SFT-LLM network (step 2)
checkpoint_file = 'checkpoint/step2_checkpoint_SFT_LLM_' + str(time_stamp) + '.pkl'
checkpoint_file = 'checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl' # GPU pre-trained
checkpoint = torch.load(checkpoint_file, map_location=device)
epoch = checkpoint['epoch']
tot_time = checkpoint['tot_time']
loss = checkpoint['loss']
print('Load pre-trained SFT-LLM: \n checkpoint file: {:s}\n epoch: {:d}, time: {:.3f}min, loss={:.4f}'.format(checkpoint_file,epoch,tot_time,loss))
net_parameters = SFT_LLM_net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
context_length = net_parameters['context_length']
num_layers = net_parameters['num_layers']
padding_int = net_parameters['padding_int']
eos_int = net_parameters['eos_int']
print(' num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, context_length, num_heads, num_layers) )
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
print('num_tokens: %d, padding_int: %d, eos_int: %d\n' % (num_tokens, padding_int, eos_int))
SFT_LLMnet = SFT_LLM(num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int)
SFT_LLMnet = SFT_LLMnet.to(device)
SFT_LLMnet.load_state_dict(checkpoint['SFT_LLMnet_dict']) # load pre-trained SFT-LM network (step 2)
num_param = number_param(SFT_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )
del checkpoint

# generate new sentence of any length
def generate(LLMnet, prompt, max_length_gen_seq, Temperature):
    #LMnet.train()
    predicted_seq = torch.ones(1, max(prompt.size(0),LLMnet.context_length)).long().to(device) * LLMnet.padding # initiliaze with padding
    predicted_seq[:, -prompt.size(0):] = prompt # fill batch_predicted_seq with prompt, right-aligned
    for k in range(max_length_gen_seq):
        context = predicted_seq[:,-LLMnet.context_length:] # size=[batch_size, context_length
        H = LLMnet.token2vec(context) + LLMnet.PE_embedding(LLMnet.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
        for transformer_block in LLMnet.transformer_blocks: H = transformer_block(H) # size=(batch_size, context_length, d)
        token_scores = H[:,-1,:] # extract last token to predict the next one, size=[batch_size, d]
        token_scores = LLMnet.token_prediction(token_scores) # compute scores, size=[batch_size, num_tokens]
        token_probs = torch.softmax(token_scores/Temperature, dim=1) # compute probs, size=[batch_size, num_tokens]
        next_token = torch.multinomial(token_probs, num_samples=1) # sample next token, size=[batch_size, 1]
        #next_token = torch.max(token_probs, dim=1).indices[0].view(1,1) # size=(1,1)
        if next_token==LLMnet.eos:
            break
        predicted_seq = torch.cat((predicted_seq, next_token), dim=1) # size=[batch_size, current_seq_len+1]
    gen_seq = predicted_seq[0][max(prompt.size(0),LLMnet.context_length):]
    return gen_seq


device: cpu 

Load pre-trained SFT-LLM: 
 checkpoint file: checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl
 epoch: 0, time: 748.458min, loss=0.0258
 num_tokens: 129, d: 384, context_length: 40, num_heads: 6, num_layers: 6

num_tokens: 129, padding_int: 127, eos_int: 128

num_net_parameters: 10761345 / 10.76 million



## Generate training set of (prompt, positive response, negative response) with rewards

### Use pre-trained SFT-LLM network (from Step 2)


In [5]:
# generate arithmetic series
max_value = 100 # maximum value in the sequence
def arithmetic_series(max_value, s, d, n):
    seq = []
    for i in range(n):
        v = s + i * d
        if v <= max_value:
            seq.append(v)
        else:
            break
    return seq

# Generate training data :
#
#  for each prompt, we collect one positive response and the associated reward
#                          and one negative response and the associated reward
#
# The response is generated auto-regressively with the pre-trained LLM of step 2
#
# Two responses are sampled and the positive response is the one with the largest reward value
#
# Reward is defined as r = r_min + ( r_max - r_min ) / (  1 + beta * || exact_response - generated_response ||^alpha )
#                          with r_min=1 (worst), r_max=7 (best), beta=0.1, alpha=0.9
#
# Training data structure :
# 
#  list_positive_responses = [ [prompt_1 + positive_response_1], [positive_reward_1]
#                              [prompt_2 + positive_response_2], [positive_reward_2]
#                                ...
#                              [prompt_N + positive_response_N], [positive_reward_N] ]
#
#  list_negative_responses = [ [prompt_1 + negative_response_1], [negative_reward_1]
#                              [prompt_2 + negative_response_2], [negative_reward_2]
#                                ...
#                              [prompt_N + negative_response_N], [negative_reward_N] ]

save_training_data = False
save_training_data = True
if save_training_data:

    # collect "human" training set
    list_positive_response = [] # list of prompts + positive responses
    list_negative_response = [] # list of prompts + negative responses
    num_training_data = 10 # debug
    # num_training_data = 100000 # 100K number of pairs of (prompt, positive response) and (prompt, negative response)
    print('num_training_data:', num_training_data)
    start = time.time()
    num_data = 0
    num_iterations = 0
    start = time.time()
    while num_data < num_training_data:

        # parameters for arithmetic series
        s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
        d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
        n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
        #print('max_value: %d, start_value: %d, common_difference: %d, number_of_terms: %d' % (max_value,s,d,n))

        # generate prompt : sample a prompt between 3 candidate prompts
        prompt = {}
        prompt[1] = 'generate an arithmetic series with ' + str(n) + ' terms starting with value ' + str(s) + ' and common difference ' + str(d)
        prompt[2] = 'make a series of arithmetic type which starts at ' + str(s) + ' with ' + str(n) + ' elements and ' + str(d) + ' common difference value'
        prompt[3] = 'Let ' + str(n) + ' be the number of terms ' + str(s) + ' the starting number and ' + str(d) + ' the common difference then write the arithmetic series'
        random_int = torch.randint(low=1, high=3+1, size=(1,)).item() # random number in {1,2,3}
        prompt_str = prompt[random_int] # e.g. generate an arithmetic series with 8 terms starting with value 36
        # print('prompt         :',prompt_str)
        prompt_ind_pytorch = torch.tensor(func_tokens2indices(func_str2tokens(prompt_str))) # convert str to pytorch indices, e.g. [102, 103, 104, 105, 106, ..]
        prompt_seq1_ind_pytorch = prompt_ind_pytorch; prompt_seq2_ind_pytorch = prompt_ind_pytorch # initializing prompt+response

        # exact response
        exact_response_token_pytorch = torch.tensor(arithmetic_series(max_value,s,d,n)) # tensor([36, 40, 44, 48, 52, ..])
    
        # generate two responses
        Temperature = 4
        max_length_gen_seq = 20
        gen_seq1_ind_pytorch = generate(SFT_LLMnet, prompt_ind_pytorch, max_length_gen_seq, Temperature) # sample one response
        gen_seq2_ind_pytorch = generate(SFT_LLMnet, prompt_ind_pytorch, max_length_gen_seq, Temperature) # sample another response
        gen_seq1_ind_pytorch = gen_seq1_ind_pytorch.to('cpu')
        gen_seq2_ind_pytorch = gen_seq2_ind_pytorch.to('cpu')

        # remove non-integer tokens, i.e. word-tokens
        gen_seq1_token = func_indices2tokens(gen_seq1_ind_pytorch.tolist()) # convert to tokens, e.g. ['90', '46', 'EOS', '71']
        gen_seq1_token = [int(x) if x.isdigit() else 50 for x in gen_seq1_token] # replace word-token 'EOS' by 50
        gen_seq1_token_pytorch = torch.tensor(gen_seq1_token) # torch([90, 46, 50, 71])
        gen_seq2_token = func_indices2tokens(gen_seq2_ind_pytorch.tolist()) 
        gen_seq2_token = [int(x) if x.isdigit() else 50 for x in gen_seq2_token] 
        gen_seq2_token_pytorch = torch.tensor(gen_seq2_token)
        # print('exact_response_token_pytorch', exact_response_token_pytorch)
        # print('gen_seq1_token_pytorch', gen_seq1_token_pytorch)
        # print('gen_seq2_token_pytorch', gen_seq2_token_pytorch)
        
        if gen_seq1_ind_pytorch.size(0)>0 and gen_seq2_ind_pytorch.size(0)>0: # generated sequences have integer tokens, otherwise go to next generation

            # concatenate prompt + generated sequences
            prompt_seq1_ind_pytorch = torch.cat( (prompt_ind_pytorch, gen_seq1_ind_pytorch) ) # concatenate prompt + seq1
            prompt_seq2_ind_pytorch = torch.cat( (prompt_ind_pytorch, gen_seq2_ind_pytorch) ) # concatenate prompt + seq2
            r_min = 1; r_max = 7
            exact_response_token_pytorch_ref = exact_response_token_pytorch
            max_size = max(exact_response_token_pytorch_ref.size(0), gen_seq1_token_pytorch.size(0)) # e.g. 4
            if len(gen_seq1_token_pytorch)>len(exact_response_token_pytorch_ref): # penalize more the reward = 1, no change to the generated_response
                exact_response_token_pytorch_ref = nn.functional.pad(exact_response_token_pytorch_ref,(0, max_size-exact_response_token_pytorch_ref.size(0)), 'constant', 1e3) 
            else: # no change to the exact_response
                gen_seq1_token_pytorch = nn.functional.pad(gen_seq1_token_pytorch,(0, max_size-gen_seq1_token_pytorch.size(0)), 'constant', 50)
            reward1 = r_min + (r_max - r_min) * ( 1 + 0.1*( (exact_response_token_pytorch_ref - gen_seq1_token_pytorch).abs() ).float().sum()**(0.5) )**(-1) 
            exact_response_token_pytorch_ref = exact_response_token_pytorch
            max_size = max(exact_response_token_pytorch_ref.size(0), gen_seq2_token_pytorch.size(0)) # e.g. 4
            if len(gen_seq2_token_pytorch)>len(exact_response_token_pytorch_ref): # penalize more the reward = 1, no change to the generated_response
                exact_response_token_pytorch_ref = nn.functional.pad(exact_response_token_pytorch_ref,(0, max_size-exact_response_token_pytorch_ref.size(0)), 'constant', 1e3) 
            else: # no change to the exact_response
                gen_seq2_token_pytorch = nn.functional.pad(gen_seq2_token_pytorch,(0, max_size-gen_seq2_token_pytorch.size(0)), 'constant', 50)
            reward2 = r_min + (r_max - r_min) * ( 1 + 0.1*( (exact_response_token_pytorch_ref - gen_seq2_token_pytorch).abs() ).float().sum()**(0.9) )**(-1) 
            # print('reward #1      : %.2f' % reward1.item() )
            # print('reward #2      : %.2f' % reward2.item() )
            
            # add samples to training dataset if reward1 is different from reward2
            if (reward1-reward2).abs()>1e-2: 
                num_data += 1
                if reward1 < reward2:
                    reward1_tmp = reward1; reward1 = reward2; reward2 = reward1_tmp
                    prompt_seq1_ind_pytorch_tmp = prompt_seq1_ind_pytorch
                    prompt_seq1_ind_pytorch = prompt_seq2_ind_pytorch
                    prompt_seq2_ind_pytorch = prompt_seq1_ind_pytorch_tmp
                    gen_seq1_token_pytorch_tmp = gen_seq1_token_pytorch
                    gen_seq1_token_pytorch = gen_seq2_token_pytorch
                    gen_seq2_token_pytorch = gen_seq1_token_pytorch_tmp 
                list_positive_response.append([prompt_seq1_ind_pytorch, reward1])
                list_negative_response.append([prompt_seq2_ind_pytorch, reward2])

            # print
            if not num_iterations%500: # 2 (debug), 1000 (GPU)
                print('num_iterations: %d, num_data: %d, time(min): %.3f' % (num_iterations, num_data, (time.time()-start)/60) )
                prompt_str = func_tokens2str(func_indices2tokens(prompt_ind_pytorch.tolist()))
                exact_response_str = exact_response_token_pytorch.tolist()
                seq1_str = func_tokens2str(func_indices2tokens(prompt_seq1_ind_pytorch.tolist()))
                seq2_str = func_tokens2str(func_indices2tokens(prompt_seq2_ind_pytorch.tolist()))
                print('prompt         :', prompt_str)
                print('exact_response :', exact_response_str)
                print('seq1           :', seq1_str)
                print('seq2           :', seq2_str)
                print('reward #1      : %.2f' % reward1.item() )
                print('reward #2      : %.2f' % reward2.item() )
            num_iterations += 1

    # print
    print('Total time(min): %.3f' % ((time.time()-start)/60) )
    print('\nnumber of training data (prompt, positive response, negative response) :',len(list_positive_response),'\n')
    for idx, (positive, negative) in enumerate(zip(list_positive_response[:3],list_negative_response[:3])):
        pos_response = func_tokens2str(func_indices2tokens(positive[0].tolist()))
        neg_response = func_tokens2str(func_indices2tokens(negative[0].tolist()))
        pos_reward = positive[1].item()
        neg_reward = negative[1].item()
        print('training_set[%d]: ' % idx )
        print('  pos_response : %s, pos_reward : %.2f ' % (pos_response, pos_reward) )
        print('  neg_response : %s, neg_reward : %.2f ' % (neg_response, neg_reward), '\n' )

    # save training data
    save_file = data_dir + '/step3_SLRM_training_set_' + time_stamp + '.pt'
    print('save_file:', save_file, '\n')
    torch.save([list_positive_response, list_negative_response],save_file) # save list of positive and negative responses

else:

    # load training data
    load_file = data_dir + '/step3_SLRM_training_set_' + time_stamp + '.pt' 
    load_file = 'dataset/step3_SLRM_training_set_26-02-07--18-48-32_100K.pt' 
    print('load_file:', load_file, '\n')
    list_positive_response, list_negative_response = torch.load(load_file) # load list of positive and negative responses

    # print
    print('number of training data (prompt, positive response, negative response) :',len(list_positive_response),'\n')
    for idx, (positive, negative) in enumerate(zip(list_positive_response[:3],list_negative_response[:3])):
        pos_response = func_tokens2str(func_indices2tokens(positive[0].tolist()))
        neg_response = func_tokens2str(func_indices2tokens(negative[0].tolist()))
        pos_reward = positive[1].item()
        neg_reward = negative[1].item()
        print('training_set[%d]: ' % idx )
        print('  pos_response : %s, pos_reward : %.2f ' % (pos_response, pos_reward) )
        print('  neg_response : %s, neg_reward : %.2f ' % (neg_response, neg_reward), '\n' )

# Total time(min): 142.362
# number of training data (prompt, positive response, negative response) : 100000 
# training_set[1]: 
#   pos_response : generate an arithmetic series with 12 terms starting with value 32 and common difference 1 36 35 36 35 36 39 38 39 42 41 44 39, pos_reward : 5.21 
#   neg_response : generate an arithmetic series with 12 terms starting with value 32 and common difference 1 32 33 34 38 36 33 38 36 39 41 40 39, neg_reward : 3.63 


num_training_data: 10
num_iterations: 0, num_data: 1, time(min): 0.004
prompt         : generate an arithmetic series with 12 terms starting with value 15 and common difference 7
exact_response : [15, 22, 29, 36, 43, 50, 57, 64, 71, 78, 85, 92]
seq1           : generate an arithmetic series with 12 terms starting with value 15 and common difference 7 15 23 31 42 49 56 57 40 83 90 91
seq2           : generate an arithmetic series with 12 terms starting with value 15 and common difference 7 15 22 29 38 43 48 51 76 75 78 87 94
reward #1      : 3.88
reward #2      : 2.91
Total time(min): 0.036

number of training data (prompt, positive response, negative response) : 10 

training_set[0]: 
  pos_response : generate an arithmetic series with 12 terms starting with value 15 and common difference 7 15 23 31 42 49 56 57 40 83 90 91, pos_reward : 3.88 
  neg_response : generate an arithmetic series with 12 terms starting with value 15 and common difference 7 15 22 29 38 43 48 51 76 75 78 87 94, 

## Split dataset 90% train set and 10% validation set

Supervised learning can overfit. Hence, we use early stopping, i.e. training is stopped if validation loss increases.

In [6]:
num_prompt_response = len(list_positive_response) # number of prompt+response sequences
num_train_prompt_response = int(0.90 * num_prompt_response)
num_val_prompt_response = num_prompt_response - num_train_prompt_response
idx_rand_perm = torch.randperm(num_prompt_response)
list_positive_response_val = list_positive_response[-num_val_prompt_response:]
list_negative_response_val = list_negative_response[-num_val_prompt_response:]
list_positive_response = list_positive_response[:num_train_prompt_response]
list_negative_response = list_negative_response[:num_train_prompt_response]
print(f'num_prompt_response: {num_prompt_response}, num_train_prompt_response: {num_train_prompt_response}, num_val_prompt_response: {num_val_prompt_response} ')


num_prompt_response: 100000, num_train_prompt_response: 90000, num_val_prompt_response: 10000 


## Train reward model with supervised learning

## Use pre-trained LLM model from Step 2 with frozen layers as the backbone

## Dataset is composed of (positive response, negative response)

In [7]:
torch.manual_seed(0) # use same initial seed for reproducibility

# Compute validation loss
def compute_val_loss(SL_RMnet, batch_size, list_positive_response, list_negative_response_val):
    SL_RMnet.eval()
    running_loss = 0.0
    num_val_prompt_response = len(list_positive_response)
    num_batch = num_val_prompt_response // batch_size
    list_prompt_response_idx = torch.arange(num_val_prompt_response).to(device) # initialize the list of (pos_response, neg_response) indices
    with torch.no_grad():
         indices = torch.arange(num_val_prompt_response) # Shuffle once per epoch
         for start_idx in range(0, num_val_prompt_response, batch_size):
            batch_idx = indices[start_idx : start_idx + batch_size]
            reward_scores = SL_RMnet(batch_idx.to(device), list_positive_response, list_negative_response) # predict rewards, size=[2*batch_size, 1]
            reward_labels = [ [list_positive_response[idx][1],list_negative_response[idx][1]] for idx in batch_idx ]
            reward_labels = torch.tensor(reward_labels).view(2*batch_size,1).to(device) # size=[2*batch_size, 1]
            diff_rewards = reward_scores[0:2*batch_size:2,:] - reward_scores[1:2*batch_size+1:2,:] # difference of rewards, sise=[batch_size, 1]
            loss_rank = - torch.log(torch.sigmoid(diff_rewards)).mean() # rank loss
            loss_mse = nn.MSELoss()(reward_scores, reward_labels) # regression loss for rewards
            loss = 1 * loss_mse + 1/4 * loss_rank # rating + rank
            running_loss += loss.detach().cpu().item()
    SL_RMnet.train()
    val_loss = running_loss / num_batch
    return val_loss 

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# GPU training
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training <==
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation <==
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length))==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

# Predict positive reward given context = { prompt + positive response }
#         negative reward given context = { prompt + negative response }
#
# Prediction is done into 2 stages :
#         1. compute self-attention between last token of the sequence and the context using pre-trained SFT-LLM (Step 2)
#         2. apply small MLP to predict scalar reward
#
# Example of reward prediction for a BATCH of sequences = (prompt + positive/negative response) 
#
# Prepare a batch of sampled (prompt+response)
#  Let the token P = <Padding>
#
#                    context_length = 7 (all prompt+response have the same context length with padding if needed) 
#                 ---------------------
# batch_seq    = [ P, P, 1, 2, 3, 4, 5 ]  // prompt = [1, 2, 3] + positive response = [4, 5]
#              = [ 1, 2, 3, 4, 7, 8, 1 ]  // prompt = [1, 2, 3] + negative response = [4, 7, 8, 1]
#                         ...
#              = [ P, P, 5, 6, 7, 8, 9 ]  // prompt = [5, 6] + positive response = [7, 8, 9]
#              = [ P, P, P, 5, 6, 3, 4 ]  // prompt = [5, 6] + negative response = [3, 4]
#                        -------------
#                                    | <= compute self-attention between last token = 4 and context = [5, 6, 3]
#                                    | <= then apply MLP to predict reward
# batch_reward_scores                = [ 5.3 ] // predicted positive reward for [1, 2, 3, 4, 5]
#                                    = [ 2.9 ] // predicted negative reward for [1, 2, 3, 4, 7, 8, 1]
#                                        ...
#                                    = [ 6.2 ] // predicted positive reward for [5, 6, 7, 8, 9]
#                                    = [ 1.8 ] // predicted negative reward for [5, 6, 3, 4]
        
# Supervised learning network for reward (Step 3)
class SL_RM(nn.Module):
    def __init__(self, SFT_LLM, d, context_length, padding_int, eos_int):
        super().__init__()
        self.SFT_LLM = SFT_LLM # token embedding layer
        self.reward_prediction = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,1)) # reward prediction layer
        self.cls_token_input = torch.tensor([0]).long().to(device)
        self.cls_token_embed = nn.Sequential( nn.Embedding(1, 4*d), nn.ReLU(), nn.Linear(4*d, d) )
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
    def forward(self, batch_idx, list_positive_response, list_negative_response): # batch_idx.size=[batch_size], len(list_prompt,list_response) =[num_prompt_response]
        pos_responses = [list_positive_response[idx][0] for idx in batch_idx] # sample list of pos_responses, len(pos_responses)=num_prompt_response
        len_pos_response = max([len(response) for response in pos_responses]) # compute max of pos_response lengths, e.g. 30=max(23,30,29)
        neg_responses = [list_negative_response[idx][0] for idx in batch_idx] # sample list of neg_responses, len(neg_responses)=num_prompt_response
        len_neg_response = max([len(response) for response in neg_responses]) # compute max of neg_response lengths, e.g. 29=max(23,29,24)
        len_response = max(len_pos_response, len_neg_response) # compute max of pos_response and neg_response lengths
        batch_size = batch_idx.size(0) # e.g. 3
        batch_seq = torch.ones(2* batch_size, max(len_response, self.context_length)).long().to(device) * self.padding # context, initialize with padding, size=[2* batch_size, context_length]
        for idx in range(batch_size): batch_seq[2*idx, -pos_responses[idx].size(0):] = pos_responses[idx] # fill context with pos_responses, right-aligned
        for idx in range(batch_size): batch_seq[2*idx+1, -neg_responses[idx].size(0):] = neg_responses[idx] # fill context with neg_responses, right-aligned
        batch_seq = batch_seq[:,-self.context_length:] # satisfy the context_length, e.g. [6, context_length=20]
        H = self.SFT_LLM.token2vec(batch_seq) + self.SFT_LLM.PE_embedding(self.SFT_LLM.seq_pos_encoding[:batch_seq.size(1)]).unsqueeze(0) # size=[2* batch_size, context_length, d]
        h_cls = self.cls_token_embed(self.cls_token_input).expand(2* batch_size,-1,-1) # [2* batch_size, 1, d]  
        H = torch.cat([H, h_cls], dim=1) # [2* batch_size, context_length+1, d]
        for transformer_block in self.SFT_LLM.transformer_blocks: H = transformer_block(H) # size=[2* batch_size, context_length+1, d)
        token_scores = H[:,-1,:] # extract last token scores to predict reward score, size=[2* batch_size, d]
        reward_scores = self.reward_prediction(token_scores) # compute reward scores, size=[2* batch_size, 1]
        return reward_scores

# use parameters of pre-trained SFT-LLM network (step 2) for SL_RM network
print('Parameters of pre-trained SFT-LLM network (step 2)')
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
context_length = net_parameters['context_length']
num_layers = net_parameters['num_layers']
padding_int = net_parameters['padding_int']
eos_int = net_parameters['eos_int']
print(' num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d' % (num_tokens, d, context_length, num_heads, num_layers) )
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
print(' num_tokens: %d, padding_int: %d, eos_int: %d\n' % (num_tokens, padding_int, eos_int))

# SL_RM network
SL_RMnet = SL_RM(SFT_LLMnet, d, context_length, padding_int, eos_int)
SL_RMnet = SL_RMnet.to(device)
num_param = number_param(SL_RMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Freeze the first transformer blocks of SFT-LLM (step 2) and unfreeze the last transformer blocks 
num_blocks_to_unfreeze = 4
blocks_to_unfreeze = [f'transformer_blocks.{num_layers - i}' for i in range(1, num_blocks_to_unfreeze + 1)]
for name, param in SL_RMnet.named_parameters():
    if 'SFT_LLM' in name:
        # Check if the parameter's name contains any of the targeted block names
        if any(block in name for block in blocks_to_unfreeze): 
            param.requires_grad = True # Let these blocks learn
        else:
            param.requires_grad = False # Freeze earlier layers

def count_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params
num_trainable = count_trainable_parameters(SL_RMnet)
print(f"Total Trainable Parameters: {num_trainable:,}")
print(f"Total Parameters (incl. frozen): {sum(p.numel() for p in SL_RMnet.parameters()):,}")

# batching parameters
num_prompt_response = len(list_positive_response) # number of prompt+response sequences
batch_size = 3 # debug
batch_size = 100 # batch size

num_batch = num_prompt_response // batch_size # number of batches
print('num_prompt_response: %d, batch_size: %d, num_batch: %d\n' % (num_prompt_response, batch_size, num_batch))

num_epochs = 30 # estimation
print('num_epochs: ',num_epochs,'\n') 

# optimizer
init_lr = 1e-4 
final_lr = 1e-5 
warmup_steps = 100
total_steps = num_batch * num_epochs  # Total training steps (e.g., 1 epoch)
# Optimizer
optimizer = torch.optim.AdamW(SL_RMnet.parameters(), lr=init_lr)

def lr_lambda(current_step):
    # 1. Linear Warmup Phase
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    # 2. Cosine Annealing Phase
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    # Clamp progress at 1.0 to avoid errors if training goes longer than expected
    progress = min(1.0, progress)
    cosine_decay = 0.5 * (1.0 + torch.cos(torch.tensor(torch.pi * progress)).item()) # Standard cosine
    # Calculate the ratio between final_lr and init_lr to ensure we hit the floor
    lr_ratio = final_lr / init_lr
    # The multiplier starts at 1.0 and ends at lr_ratio
    return lr_ratio + (1.0 - lr_ratio) * cosine_decay
# Scheduler
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.95, patience=5, verbose=True) # TRY


# save checkpoint
net_parameters = SFT_LLM_net_parameters
checkpoint_dir = os.path.join("checkpoint")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
print('checkpoint file :', checkpoint_dir + '/step3_checkpoint_SL_RM_' + time_stamp + '.pkl', '\n')

# Understanding the loss
#
# 1. loss_rewards = MSE( predicted_reward, label_reward )
#
# reward_scores                  = [ 5.3 ] // predicted positive reward for seq_1 <= index 2*i   for pos, i = 0, 1, ..., num_prompts-1
#   = predicted_reward           = [ 2.9 ] // predicted negative reward for seq_1 <= index 2*i+1 for neg, i = 0, 1, ..., num_prompts-1
#                                    ...
#                                = [ 6.2 ] // predicted Positive reward for seq_B
#                                = [ 1.8 ] // predicted Negative reward for seq_B
#
# 2. loss_rank = - log( sigmoid( positive_reward - negative_reward ) )
#
# 3. total_loss = loss_rewards + cst * loss_rank

# Train network to predict reward from (pos_response, neg_response)
start = time.time()
continue_training = True; epoch = 0
best_val_loss = 1e10; patience_counter = 0; patience_limit = 5 # Stop after 4 consecutive larger validations
while continue_training:
    epoch += 1
    indices = torch.randperm(num_prompt_response) # Shuffle once per epoch
    running_loss = 0.0 # tracking total loss value
    idx_batch = 0
    for start_idx in range(0, num_prompt_response, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]
        batch_size_idx = len(batch_idx)
        idx_batch += 1   
        reward_scores = SL_RMnet(batch_idx.to(device), list_positive_response, list_negative_response) # predict rewards, size=[2*batch_size, 1]
        reward_labels = [ [list_positive_response[idx][1],list_negative_response[idx][1]] for idx in batch_idx ]
        reward_labels = torch.tensor(reward_labels).view(2*batch_size_idx,1).to(device) # size=[2*batch_size, 1]
        diff_rewards = reward_scores[0:2*batch_size_idx:2,:] - reward_scores[1:2*batch_size_idx+1:2,:] # difference of rewards, sise=[batch_size, 1]
        loss_rank = - torch.log(torch.sigmoid(diff_rewards)).mean() # rank loss
        loss_mse = nn.MSELoss()(reward_scores, reward_labels) # regression loss for rewards
        loss = 1 * loss_mse + 1/4 * loss_rank # rating + rank
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(SL_RMnet.parameters(), 1.0) # grad_norm_clip=1.0
        optimizer.step()
        scheduler.step()
    loss_epoch = running_loss / num_batch
    
    # Compute validation loss
    val_loss = compute_val_loss(SL_RMnet, batch_size, list_positive_response_val, list_negative_response_val)
    print(f'epoch : {epoch}, lr : {optimizer.param_groups[0]["lr"]:.7f}, val_loss : {val_loss:.4f}, patience : {patience_counter}, time(min) : {(time.time()-start)/60:.3f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_counter = 0  # Reset counter to 0
        # save checkpoint
        torch.save({
            'epoch': epoch,
            'tot_time': time.time()-start,
            'best_val_loss': best_val_loss,
            'net_parameters': net_parameters,
            'SL_RMnet_dict': SL_RMnet.state_dict(),
            }, '{}.pkl'.format(checkpoint_dir + "/step3_checkpoint_SL_RM_" + time_stamp ))
    else: # val_loss does not improve
        patience_counter += 1
        # Stopping condition
        if patience_counter >= patience_limit:
            print(f'\nEarly stopping triggered -- training stopped\n')
            continue_training = False
            break # Break the batch loop     
                    
    if not epoch%1:
        print('\nEpoch: %d, time(min): %.3f, lr= %.6f, loss_epoch: %.3f, best_val_loss: %.3f' % (epoch, (time.time()-start)/60, optimizer.param_groups[0]['lr'], loss_epoch, best_val_loss) )
        # print rewards
        idx_prompt = 0
        print('pos response token :',func_tokens2str(func_indices2tokens(list_positive_response[batch_idx[idx_prompt]][0].tolist())))
        print('pos reward label   :',reward_labels[2*idx_prompt,:].squeeze().item())
        print('pos reward pred    :',reward_scores[2*idx_prompt,:].squeeze().item())
        print('neg response token :',func_tokens2str(func_indices2tokens(list_negative_response[batch_idx[idx_prompt]][0].tolist())))
        print('neg reward label   :',reward_labels[2*idx_prompt+1,:].squeeze().item())
        print('neg reward pred    :',reward_scores[2*idx_prompt+1,:].squeeze().item(),'\n')


# Early stopping triggered -- training stopped
# epoch : 26, lr : 0.0000139, val_loss : 0.6749, patience : 4, time(min) : 26.499



NVIDIA RTX A5000
device: cuda 

Parameters of pre-trained SFT-LLM network (step 2)
 num_tokens: 129, d: 384, context_length: 40, num_heads: 6, num_layers: 6
 num_tokens: 129, padding_int: 127, eos_int: 128

num_net_parameters: 11945986 / 11.95 million

Total Trainable Parameters: 8,282,497
Total Parameters (incl. frozen): 11,945,986
num_prompt_response: 90000, batch_size: 100, num_batch: 900

num_epochs:  30 

checkpoint file : checkpoint/step3_checkpoint_SL_RM_26-02-07--18-48-32.pkl 

epoch : 1, lr : 0.0000998, val_loss : 1.3404, patience : 0, time(min) : 1.028

Epoch: 1, time(min): 1.030, lr= 0.000100, loss_epoch: 2.047, best_val_loss: 1.340
pos response token : generate an arithmetic series with 12 terms starting with value 68 and common difference 5 68 75 78 83 86 90 94
pos reward label   : 5.505645751953125
pos reward pred    : 4.864037990570068
neg response token : generate an arithmetic series with 12 terms starting with value 68 and common difference 5 68 73 76 85 94 99
neg rew

## Load pre-trained SL-RM network


In [7]:
import torch
import torch.nn as nn

# GPU training
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training <==
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation <==
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length))==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

# Supervised learning network for reward (Step 3)
class SL_RM(nn.Module):
    def __init__(self, SFT_LLM, d, context_length, padding_int, eos_int):
        super().__init__()
        self.SFT_LLM = SFT_LLM # token embedding layer
        self.reward_prediction = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,1)) # reward prediction layer
        self.cls_token_input = torch.tensor([0]).long().to(device)
        self.cls_token_embed = nn.Sequential( nn.Embedding(1, 4*d), nn.ReLU(), nn.Linear(4*d, d) )
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
    def forward(self, batch_idx, list_positive_response, list_negative_response): # batch_idx.size=[batch_size], len(list_prompt,list_response) =[num_prompt_response]
        pos_responses = [list_positive_response[idx][0] for idx in batch_idx] # sample list of pos_responses, len(pos_responses)=num_prompt_response
        len_pos_response = max([len(response) for response in pos_responses]) # compute max of pos_response lengths, e.g. 30=max(23,30,29)
        neg_responses = [list_negative_response[idx][0] for idx in batch_idx] # sample list of neg_responses, len(neg_responses)=num_prompt_response
        len_neg_response = max([len(response) for response in neg_responses]) # compute max of neg_response lengths, e.g. 29=max(23,29,24)
        len_response = max(len_pos_response, len_neg_response) # compute max of pos_response and neg_response lengths
        batch_size = batch_idx.size(0) # e.g. 3
        batch_seq = torch.ones(2* batch_size, max(len_response, self.context_length)).long().to(device) * self.padding # context, initialize with padding, size=[2* batch_size, context_length]
        for idx in range(batch_size): batch_seq[2*idx, -pos_responses[idx].size(0):] = pos_responses[idx] # fill context with pos_responses, right-aligned
        for idx in range(batch_size): batch_seq[2*idx+1, -neg_responses[idx].size(0):] = neg_responses[idx] # fill context with neg_responses, right-aligned
        batch_seq = batch_seq[:,-self.context_length:] # satisfy the context_length, e.g. [6, context_length=20]
        H = self.SFT_LLM.token2vec(batch_seq) + self.SFT_LLM.PE_embedding(self.SFT_LLM.seq_pos_encoding[:batch_seq.size(1)]).unsqueeze(0) # size=[2* batch_size, context_length, d]
        h_cls = self.cls_token_embed(self.cls_token_input).expand(2* batch_size,-1,-1) # [2* batch_size, 1, d]  
        H = torch.cat([H, h_cls], dim=1) # [2* batch_size, context_length+1, d]
        for transformer_block in self.SFT_LLM.transformer_blocks: H = transformer_block(H) # size=[2* batch_size, context_length+1, d)
        token_scores = H[:,-1,:] # extract last token scores to predict reward score, size=[2* batch_size, d]
        reward_scores = self.reward_prediction(token_scores) # compute reward scores, size=[2* batch_size, 1]
        return reward_scores

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

file_dictionary = 'dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt' # GPU pre-trained
print('file_dictionary:', file_dictionary, '\n')
import os
os.makedirs('dataset', exist_ok=True)
if not os.path.exists(file_dictionary):
    print(f'Downloading {file_dictionary} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/jvb5xkk0kfzwrm7vzha3o/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt?rlkey=tprax7nqqpvogo04fhe7m2okd&dl=1"
    !wget -q "{dropbox_url}" -O "{file_dictionary}"
dictionary, num_tokens, token2index, index2token = torch.load(file_dictionary, map_location=device) # load dictionary of tokens
print('dictionary:',dictionary,'\n')
print('num_tokens (unique):',num_tokens,'\n')
print('token2index:', token2index,'\n')
print('index2token:', index2token,'\n')
func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'

# pre-trained SL-RM network
# # checkpoint_file = checkpoint_dir + '/step3_checkpoint_SL_RM_' + time_stamp + '.pkl'
# # checkpoint_file = 'checkpoint/step3_checkpoint_SL_RM_26-02-07--18-48-32.pkl'
import os
checkpoint_file = 'checkpoint/step3_checkpoint_SL_RM_26-02-07--18-48-32_100K_10M.pkl' # GPU pre-trained
print('checkpoint_file:', checkpoint_file, '\n')
os.makedirs('checkpoint', exist_ok=True)
if not os.path.exists(checkpoint_file):
    print(f'Downloading {checkpoint_file} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/8njbmywnqub4nt7x6o63j/step3_checkpoint_SL_RM_26-02-07-18-48-32_100K_10M.pkl?rlkey=eprvslw7w5zdny9ujf5svzp21&dl=1"
    !wget -q "{dropbox_url}" -O "{checkpoint_file}"
checkpoint = torch.load(checkpoint_file, map_location=device)
epoch = checkpoint['epoch']
tot_time = checkpoint['tot_time']
best_val_loss = checkpoint['best_val_loss']
print('Load pre-trained SFT-LM: \n checkpoint file: {:s}\n epoch: {:d}, time: {:.3f}min, best_val_loss={:.4f}'.format(checkpoint_file,epoch,tot_time,best_val_loss))
net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
context_length = net_parameters['context_length']
num_layers = net_parameters['num_layers']
padding_int = net_parameters['padding_int']
eos_int = net_parameters['eos_int']
print(' num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, context_length, num_heads, num_layers) )
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
print('num_tokens: %d, padding_int: %d, eos_int: %d\n' % (num_tokens, padding_int, eos_int))
SL_RMnet = SL_RM(SFT_LLMnet, d, context_length, padding_int, eos_int)
SL_RMnet = SL_RMnet.to(device)
SL_RMnet.load_state_dict(checkpoint['SL_RMnet_dict']) # load pre-trained SL-RM network from step #3
num_param = number_param(SL_RMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )
del checkpoint


# print (generate a few training data to run the examples below)
batch_idx = torch.randperm(len(list_positive_response))[:5] # select 3 indices
reward_scores = SL_RMnet(batch_idx.to(device), list_positive_response, list_negative_response) # predict rewards, size=[2*3, 1]
reward_labels = [ [list_positive_response[idx][1],list_negative_response[idx][1]] for idx in batch_idx ]
reward_labels = torch.tensor(reward_labels).view(2*batch_idx.size(0),1) # size=[2*3, 1]
for idx in range(batch_idx.size(0)):
    print('pos response token :',func_tokens2str(func_indices2tokens(list_positive_response[batch_idx[idx]][0].tolist())))
    print('pos reward label   :',reward_labels[2*idx,:].squeeze().item())
    print('pos reward pred    :',reward_scores[2*idx,:].squeeze().item())
    print('neg response token :',func_tokens2str(func_indices2tokens(list_negative_response[batch_idx[idx]][0].tolist())))
    print('neg reward label   :',reward_labels[2*idx+1,:].squeeze().item())
    print('neg reward pred    :',reward_scores[2*idx+1,:].squeeze().item(),'\n')


device: cpu 

file_dictionary: dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt 

dictionary: ['51', '57', '63', '69', '75', '81', '87', '93', '99', '<SEP>', '91', '1', '2', '3', '4', '5', '6', '7', '8', '9', '77', '79', '83', '85', '89', '48', '55', '62', '76', '90', '97', '29', '38', '47', '56', '65', '74', '92', '94', '100', '50', '64', '71', '78', '39', '95', '46', '53', '60', '67', '88', '13', '22', '31', '40', '49', '58', '41', '42', '43', '44', '30', '54', '15', '36', '72', '86', '96', '18', '19', '20', '21', '23', '24', '25', '26', '27', '28', '73', '98', '82', '52', '34', '66', '70', '68', '84', '37', '61', '11', '14', '17', '32', '59', '10', '12', '16', '80', '35', '33', '45', '0', 'generate', 'an', 'arithmetic', 'series', 'with', 'terms', 'starting', 'value', 'and', 'common', 'difference', 'Let', 'be', 'the', 'number', 'of', 'then', 'write', 'make', 'a', 'type', 'which', 'starts', 'at', 'elements', '<PAD>', '<EOS>'] 

num_tokens (unique): 129 

token2index: {'51': 0, 